# **Exploratory Data Analysis**

**Setup**

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d jessicali9530/stanford-dogs-dataset
!unzip -q -o stanford-dogs-dataset.zip -d stanford-dogs/
!echo "Done. Folder structure:"
!ls stanford-dogs/

**imports and styling**

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# blue color palette
SKY_BLUE   = '#87CEEB'   # primary
STEEL_BLUE = '#4682B4'   # secondary
LIGHT_BLUE = '#B0E0E6'   # tertiary
ACCENT     = '#5DADE2'   # accent
NAVY       = '#1B3A57'   # text/lines

PALETTE = ['#87CEEB', '#5DADE2', '#4682B4', '#85C1E9',
           '#3498DB', '#2E86C1', '#1B4F72', '#B0E0E6']

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams.update({
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
    'axes.edgecolor':     '#D5DBDB',
    'axes.labelcolor':    NAVY,
    'axes.titlecolor':    NAVY,
    'axes.titleweight':   'bold',
    'axes.titlesize':     14,
    'axes.titlepad':      14,
    'axes.labelsize':     12,
    'axes.labelweight':   'semibold',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'xtick.color':        NAVY,
    'ytick.color':        NAVY,
    'xtick.labelsize':    10,
    'ytick.labelsize':    10,
    'font.family':        'sans-serif',
    'font.sans-serif':    ['DejaVu Sans', 'Arial'],
    'font.size':          11,
    'grid.color':         '#ECF0F1',
    'grid.alpha':         0.8,
    'grid.linewidth':     0.8,
    'figure.dpi':         100,
    'savefig.dpi':        150,
    'legend.frameon':     False,
    'legend.fontsize':    10,
})
sns.set_palette(PALETTE)
print('Styling loaded. Primary color: sky blue (#87CEEB)')

**Data Summary, Schema, and Missing Values**

In [ ]:
DATA_DIR = Path('stanford-dogs/images/Images')
breed_folders = sorted([f for f in DATA_DIR.iterdir() if f.is_dir()])

records = []
for folder in breed_folders:
    # Folders look like 'n02085620-Chihuahua' — strip the prefix
    breed_name = folder.name.split('-', 1)[1].replace('_', ' ').title()
    for img_path in folder.glob('*.jpg'):
        records.append({
            'path':          str(img_path),
            'breed':         breed_name,
            'breed_folder':  folder.name,
        })

df = pd.DataFrame(records)
df.head()

**counts and schema**

In [ ]:
print('=' * 60)
print('SCHEMA')
print('=' * 60)
print(df.dtypes.to_string())

print('\n' + '=' * 60)
print('MISSING VALUES')
print('=' * 60)
print(df.isnull().sum().to_string())

print('\n' + '=' * 60)
print('RECORD COUNTS')
print('=' * 60)
counts = df['breed'].value_counts()
print(f'Total images:      {len(df):>8,}')
print(f'Total breeds:      {df["breed"].nunique():>8,}')
print(f'Avg images/breed:  {len(df) / df["breed"].nunique():>8.1f}')
print(f'Min images/breed:  {counts.min():>8,}  ({counts.idxmin()})')
print(f'Max images/breed:  {counts.max():>8,}  ({counts.idxmax()})')
print(f'Median per breed:  {counts.median():>8.1f}')

**Class distribution**

In [ ]:
breed_counts = df['breed'].value_counts()

fig, ax = plt.subplots(figsize=(13, 9))
top30 = breed_counts.head(30)
bars = ax.barh(range(len(top30)), top30.values,
               color=SKY_BLUE, edgecolor=STEEL_BLUE, linewidth=0.8)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30.index)
ax.invert_yaxis()
ax.set_xlabel('Number of Images')
ax.set_title('Top 30 Breeds by Image Count')
for i, v in enumerate(top30.values):
    ax.text(v + 2, i, str(v), va='center', fontsize=9, color=NAVY)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))
bot30 = breed_counts.tail(30).iloc[::-1]
ax.barh(range(len(bot30)), bot30.values,
        color=LIGHT_BLUE, edgecolor=STEEL_BLUE, linewidth=0.8)
ax.set_yticks(range(len(bot30)))
ax.set_yticklabels(bot30.index)
ax.invert_yaxis()
ax.set_xlabel('Number of Images')
ax.set_title('Bottom 30 Breeds by Image Count (the rare ones)')
for i, v in enumerate(bot30.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=9, color=NAVY)
plt.tight_layout()
plt.show()

**Distribution of images per breed**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(breed_counts.values, bins=25, color=SKY_BLUE,
        edgecolor=NAVY, alpha=0.85, linewidth=1)
ax.axvline(breed_counts.mean(), color=STEEL_BLUE, linestyle='--',
           linewidth=2, label=f'Mean: {breed_counts.mean():.1f}')
ax.axvline(breed_counts.median(), color=NAVY, linestyle=':',
           linewidth=2, label=f'Median: {breed_counts.median():.1f}')
ax.set_xlabel('Images per Breed')
ax.set_ylabel('Number of Breeds')
ax.set_title('Distribution of Images per Breed')
ax.legend()
plt.tight_layout()
plt.show()

**Image Properties Analysis**


Using a random sample of 2000

In [ ]:
SAMPLE_SIZE = 2000
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)


widths, heights, file_sizes = [], [], []
mean_r, mean_g, mean_b, brightness = [], [], [], []

for path in sample_df['path']:
    try:
        with Image.open(path) as img:
            w, h = img.size
            widths.append(w); heights.append(h)
            file_sizes.append(os.path.getsize(path) / 1024)  # KB
            arr = np.array(img.convert('RGB'))
            mean_r.append(arr[:, :, 0].mean())
            mean_g.append(arr[:, :, 1].mean())
            mean_b.append(arr[:, :, 2].mean())
            brightness.append(arr.mean())
    except Exception:
        for lst in [widths, heights, file_sizes, mean_r, mean_g, mean_b, brightness]:
            lst.append(np.nan)

sample_df['width']        = widths
sample_df['height']       = heights
sample_df['aspect_ratio'] = sample_df['width'] / sample_df['height']
sample_df['file_size_kb'] = file_sizes
sample_df['mean_r']       = mean_r
sample_df['mean_g']       = mean_g
sample_df['mean_b']       = mean_b
sample_df['brightness']   = brightness

sample_df[['width', 'height', 'aspect_ratio', 'file_size_kb', 'brightness']].describe().round(1)

**Image Dimensions**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].hist(sample_df['width'].dropna(), bins=40,
             color=SKY_BLUE, edgecolor=NAVY, alpha=0.85, linewidth=0.8)
axes[0].set_title('Image Width')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(sample_df['width'].median(), color=STEEL_BLUE,
                linestyle='--', label=f'Median: {sample_df["width"].median():.0f}px')
axes[0].legend()

axes[1].hist(sample_df['height'].dropna(), bins=40,
             color=STEEL_BLUE, edgecolor=NAVY, alpha=0.85, linewidth=0.8)
axes[1].set_title('Image Height')
axes[1].set_xlabel('Height (px)')
axes[1].axvline(sample_df['height'].median(), color=NAVY,
                linestyle='--', label=f'Median: {sample_df["height"].median():.0f}px')
axes[1].legend()

axes[2].hist(sample_df['aspect_ratio'].dropna(), bins=40,
             color=ACCENT, edgecolor=NAVY, alpha=0.85, linewidth=0.8)
axes[2].set_title('Aspect Ratio (W / H)')
axes[2].set_xlabel('Aspect Ratio')
axes[2].axvline(1.0, color=NAVY, linestyle=':', linewidth=2, label='Square (1.0)')
axes[2].legend()

plt.tight_layout()
plt.show()

**File Size and Color Channels**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(sample_df['file_size_kb'].dropna(), bins=40,
             color=SKY_BLUE, edgecolor=NAVY, alpha=0.85, linewidth=0.8)
axes[0].set_title('File Size Distribution')
axes[0].set_xlabel('File Size (KB)')
axes[0].set_ylabel('Frequency')

axes[1].hist(sample_df['mean_r'], bins=35, color='#E74C3C',
             alpha=0.55, label='Red',   edgecolor='white')
axes[1].hist(sample_df['mean_g'], bins=35, color='#27AE60',
             alpha=0.55, label='Green', edgecolor='white')
axes[1].hist(sample_df['mean_b'], bins=35, color=STEEL_BLUE,
             alpha=0.55, label='Blue',  edgecolor='white')
axes[1].set_title('Mean RGB Channel Intensities')
axes[1].set_xlabel('Mean Intensity (0–255)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

**Q-Q Plots**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in zip(
    axes,
    ['width', 'brightness', 'aspect_ratio'],
    ['Width', 'Brightness', 'Aspect Ratio']
):
    stats.probplot(sample_df[col].dropna(), dist='norm', plot=ax)
    ax.get_lines()[0].set_markerfacecolor(SKY_BLUE)
    ax.get_lines()[0].set_markeredgecolor(STEEL_BLUE)
    ax.get_lines()[0].set_markersize(5)
    ax.get_lines()[1].set_color(NAVY)
    ax.get_lines()[1].set_linewidth(2)
    ax.set_title(f'Q-Q Plot: {title} vs Normal')

plt.tight_layout()
plt.show()

**Correlation heatmap of image properties**

In [ ]:
numeric_cols = ['width', 'height', 'aspect_ratio', 'file_size_kb',
                'mean_r', 'mean_g', 'mean_b', 'brightness']
corr = sample_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues',
            square=True, cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
            linewidths=1, linecolor='white', annot_kws={'color': NAVY},
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Heatmap: Image Properties', pad=15)
plt.tight_layout()
plt.show()

**Width vs. Height Scatter**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(sample_df['width'], sample_df['height'],
           color=SKY_BLUE, alpha=0.45, edgecolor=STEEL_BLUE,
           s=28, linewidth=0.5)
maxd = max(sample_df['width'].max(), sample_df['height'].max())
ax.plot([0, maxd], [0, maxd], color=NAVY, linestyle='--',
        linewidth=2, label='1:1 (square)', alpha=0.7)
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.set_title('Image Width vs Height')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

**Brightness Across Top Breeds(Violin Plot)**

In [ ]:
top10_breeds = breed_counts.head(10).index.tolist()
violin_df = sample_df[sample_df['breed'].isin(top10_breeds)].dropna(subset=['brightness'])

fig, ax = plt.subplots(figsize=(15, 7))
sns.violinplot(data=violin_df, x='breed', y='brightness',
               color=SKY_BLUE, ax=ax, inner='box',
               linewidth=1.2, edgecolor=NAVY)
ax.set_title('Brightness Distribution Across Top 10 Most-Represented Breeds')
ax.set_xlabel('')
ax.set_ylabel('Mean Brightness (0–255)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

**File Size vs. Image Area**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sample_df['area_kpx'] = (sample_df['width'] * sample_df['height']) / 1000
ax.scatter(sample_df['area_kpx'], sample_df['file_size_kb'],
           color=SKY_BLUE, alpha=0.45, edgecolor=STEEL_BLUE,
           s=28, linewidth=0.5)
ax.set_xlabel('Image Area (thousands of pixels)')
ax.set_ylabel('File Size (KB)')
ax.set_title('File Size vs Image Area')
plt.tight_layout()
plt.show()

**Sample Images**

In [ ]:
rng = np.random.default_rng(42)
sample_breeds = rng.choice(df['breed'].unique(), 24, replace=False)

fig, axes = plt.subplots(4, 6, figsize=(20, 13))
fig.patch.set_facecolor('white')
for ax, breed in zip(axes.flat, sample_breeds):
    img_path = df[df['breed'] == breed]['path'].sample(1, random_state=1).iloc[0]
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(breed, fontsize=10, color=NAVY, fontweight='semibold')
    ax.axis('off')
plt.suptitle('Random Sample: One Image Per Breed (24 breeds shown)',
             fontsize=16, fontweight='bold', color=NAVY, y=1.0)
plt.tight_layout()
plt.show()

**Multiple images of the same breed**

In [ ]:
# Pick one breed and show 8 examples
focus_breed = 'Golden Retriever' if 'Golden Retriever' in df['breed'].values else df['breed'].iloc[0]
breed_imgs = df[df['breed'] == focus_breed]['path'].sample(8, random_state=7).tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.patch.set_facecolor('white')
for ax, p in zip(axes.flat, breed_imgs):
    ax.imshow(Image.open(p))
    ax.axis('off')
plt.suptitle(f'8 Random Images: {focus_breed}',
             fontsize=16, fontweight='bold', color=NAVY, y=1.0)
plt.tight_layout()
plt.show()

**Outlier Detection**

In [ ]:
def find_outliers_iqr(series, k=1.5):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return s[(s < lo) | (s > hi)]

print('Outlier counts (IQR method, k=1.5):')
print('-' * 50)
for col in ['width', 'height', 'aspect_ratio', 'file_size_kb', 'brightness']:
    out = find_outliers_iqr(sample_df[col])
    pct = 100 * len(out) / sample_df[col].notna().sum()
    print(f'  {col:<15} {len(out):>5} outliers ({pct:>4.1f}%)')

**Most Extreme aspect ratio**

In [ ]:
extreme_wide = sample_df.dropna(subset=['aspect_ratio']).nlargest(4, 'aspect_ratio')
extreme_tall = sample_df.dropna(subset=['aspect_ratio']).nsmallest(4, 'aspect_ratio')

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.patch.set_facecolor('white')

for ax, (_, row) in zip(axes[0], extreme_wide.iterrows()):
    ax.imshow(Image.open(row['path']))
    ax.set_title(f'{row["breed"]}\nAR: {row["aspect_ratio"]:.2f}',
                 fontsize=10, color=NAVY)
    ax.axis('off')

for ax, (_, row) in zip(axes[1], extreme_tall.iterrows()):
    ax.imshow(Image.open(row['path']))
    ax.set_title(f'{row["breed"]}\nAR: {row["aspect_ratio"]:.2f}',
                 fontsize=10, color=NAVY)
    ax.axis('off')

plt.suptitle('Aspect Ratio Outliers — Top: Widest    Bottom: Tallest',
             fontsize=15, fontweight='bold', color=NAVY, y=1.0)
plt.tight_layout()
plt.show()

**Smallest Images**

In [ ]:
sample_df['min_dim'] = sample_df[['width', 'height']].min(axis=1)
smallest = sample_df.dropna(subset=['min_dim']).nsmallest(8, 'min_dim')

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.patch.set_facecolor('white')
for ax, (_, row) in zip(axes.flat, smallest.iterrows()):
    ax.imshow(Image.open(row['path']))
    ax.set_title(f'{row["breed"]}\n{int(row["width"])}×{int(row["height"])}px',
                 fontsize=10, color=NAVY)
    ax.axis('off')
plt.suptitle('8 Smallest Images in the Sample',
             fontsize=15, fontweight='bold', color=NAVY, y=1.0)
plt.tight_layout()
plt.show()

**Data Summary**

**Class balance**

The largest class (Maltese Dog) has 252 images, and the smallest (Redbone) has 148, which is about a 1.7× difference. The average is 171.5 images per breed, and the median is 159.5. Most breeds are clustered around 150 to 160 images, with a smaller group above 200. This is much more balanced than real world datasets, where imbalance can be extreme.

**Image dimensions**

Most images are not random sizes. The width has a strong peak at 500 px (median = 500), and height clusters around 375 px. Aspect ratios fall into clear groups, mainly 1.33 (4:3) and 1.5 (3:2), with smaller groups at 0.67 and 0.75, which are the same ratios rotated. Very few images are square. This shows most images come from standard camera formats.

**Q-Q plots**

Width is clearly non normal, with a heavy right tail and repeated values from common image sizes. Brightness is close to normal, with points following a nearly straight line. Aspect ratio is also non normal, with visible step patterns caused by the two main ratio groups. Different features follow different distributions.

**Correlations**

Width and height have a correlation of 0.66, meaning images usually scale together. Width and height with file size are 0.71 to 0.75, which makes sense since larger images take more space. RGB channels and brightness are 0.84 to 0.96, since brightness is computed from them. Aspect ratio is mostly independent from everything else, making it a clean separate feature.

**Color cast**

Average channel values are about R ≈ 125, G ≈ 120, B ≈ 110. This shows a small red bias, giving a warm tone. This is similar to ImageNet, so pretrained models should work well.

**Brightness differences across breeds**

Afghan Hound images are the brightest with a median around 130. Bernese Mountain Dog and Great Pyrenees images are darker, around 95. Some breeds like Shih-Tzu and Pomeranian have very wide brightness ranges, meaning their images were taken under many lighting conditions.

**Outliers**

A few images are much larger than the rest, around 8 million pixels and 1MB file size, and can simply be resized. Very small images should be removed before training.

# **Preprocessing and Augmentation**

**Filter out very small images**

Before any training, drop images smaller than 100 px on the shorter side.

In [ ]:
import json
from pathlib import Path

CACHE = Path('image_dims.json')

if CACHE.exists():
    dims = json.loads(CACHE.read_text())
    print(f'Loaded cached dimensions for {len(dims):,} images.')
else:
    dims = {}
    for i, p in enumerate(df['path']):
        try:
            with Image.open(p) as img:
                dims[p] = list(img.size)  # (w, h)
        except Exception:
            dims[p] = None
        if (i + 1) % 5000 == 0:
            print(f'  {i+1:,}/{len(df):,}')
    CACHE.write_text(json.dumps(dims))
    print('Cached.')

# Attach dims to df and filter
df['width']  = df['path'].map(lambda p: dims.get(p, [None, None])[0])
df['height'] = df['path'].map(lambda p: dims.get(p, [None, None])[1])
df = df.dropna(subset=['width', 'height']).copy()
df['min_dim'] = df[['width', 'height']].min(axis=1)

before = len(df)
df = df[df['min_dim'] >= 100].reset_index(drop=True)
after = len(df)
print(f'Dropped {before - after} tiny images. Kept {after:,}.')

**Train / validation / test split**

In [ ]:
from sklearn.model_selection import train_test_split

# 70 / 15 / 15 stratified split
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['breed'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['breed'], random_state=42
)

print(f'Train:  {len(train_df):>6,}  ({len(train_df)/len(df):.0%})')
print(f'Val:    {len(val_df):>6,}  ({len(val_df)/len(df):.0%})')
print(f'Test:   {len(test_df):>6,}  ({len(test_df)/len(df):.0%})')
print(f'Total:  {len(df):>6,}')

train_df.to_csv('split_train.csv', index=False)
val_df.to_csv('split_val.csv', index=False)
test_df.to_csv('split_test.csv', index=False)


**Resize: get every image to the same size**

Models need fixed-size inputs. The naive way is `Resize((224, 224))` which forces the image into a square — but our EDA showed most photos are 4:3 or 3:2, so this **squashes** the dog.

The smart way is to first resize so the *shorter side* is 256, then crop the center 224×224. The dog keeps its real proportions.

Using one sample image:

In [ ]:
from PIL import Image
import torch
from torchvision import transforms

demo_idx   = 0
demo_path  = train_df['path'].iloc[demo_idx]
demo_breed = train_df['breed'].iloc[demo_idx]
demo_img   = Image.open(demo_path).convert('RGB')

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(demo_img)
ax.set_title(f'Original: {demo_breed}  ({demo_img.size[0]}×{demo_img.size[1]} px)',
             color=NAVY)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
naive  = transforms.Resize((224, 224))(demo_img)
smart  = transforms.CenterCrop(224)(transforms.Resize(256)(demo_img))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')

axes[0].imshow(demo_img)
axes[0].set_title('Original', color=NAVY)
axes[0].axis('off')

axes[1].imshow(naive)
axes[1].set_title('Naive Resize(224, 224)\n(squashes the dog)', color=NAVY)
axes[1].axis('off')

axes[2].imshow(smart)
axes[2].set_title('Resize(256) + CenterCrop(224)\n(keeps proportions)', color=NAVY)
axes[2].axis('off')

plt.tight_layout()
plt.show()

**Random Crops: show the model many views of the same dog**

For training, instead of always cropping the center, we crop a *random region* and resize it to 224×224. Every time the model sees this image, it sees a slightly different view. This teaches the model to recognize a dog whether it's centered, off to the side, or zoomed in.

Below: 8 different crops of the same dog. Each is what the model would see if this image came up during training.

In [ ]:
rrc = transforms.RandomResizedCrop(
    size=224,
    scale=(0.7, 1.0),       # crop covers 70-100% of the image
    ratio=(0.75, 1.33),     # matches our bimodal aspect ratios
)

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
fig.patch.set_facecolor('white')
torch.manual_seed(0)
for i, ax in enumerate(axes.flat):
    ax.imshow(rrc(demo_img))
    ax.set_title(f'Random crop #{i+1}', fontsize=10, color=NAVY)
    ax.axis('off')
plt.suptitle('Same image, 8 different random crops',
             fontsize=14, fontweight='bold', color=NAVY, y=1.0)
plt.tight_layout()
plt.show()

**Horizontal Flip: a mirrored dog is still the same dog**

In [ ]:
flipped = transforms.functional.hflip(demo_img)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.patch.set_facecolor('white')
axes[0].imshow(demo_img); axes[0].set_title('Original', color=NAVY); axes[0].axis('off')
axes[1].imshow(flipped);  axes[1].set_title('Horizontally Flipped', color=NAVY); axes[1].axis('off')
plt.tight_layout()
plt.show()

**Color Jitter: simulate different lighting conditions**

Our EDA showed brightness varies a lot across breeds — some photos are taken in bright sunlight, others in dim kennels. We don't want the model to use brightness as a shortcut to recognize a breed (e.g., "Bernese Mountain Dogs are always dark").

Color jitter randomly tweaks brightness, contrast, saturation, and hue every time. The model learns to recognize the *dog* regardless of lighting.

In [ ]:
jitter = transforms.ColorJitter(
    brightness=0.2,
    contrast=0.2,
    saturation=0.2,
    hue=0.05,
)

fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
fig.patch.set_facecolor('white')
axes[0].imshow(demo_img); axes[0].set_title('Original', color=NAVY); axes[0].axis('off')
torch.manual_seed(7)
for i, ax in enumerate(axes[1:]):
    ax.imshow(jitter(demo_img))
    ax.set_title(f'Jittered #{i+1}', fontsize=11, color=NAVY)
    ax.axis('off')
plt.tight_layout()
plt.show()

**Rotation: simulate different camera angles**

A small random rotation (±15°) makes the model less sensitive to whether the camera was held perfectly level.

In [ ]:
rotate = transforms.RandomRotation(degrees=15)

fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
fig.patch.set_facecolor('white')
axes[0].imshow(demo_img); axes[0].set_title('Original', color=NAVY); axes[0].axis('off')
torch.manual_seed(3)
for i, ax in enumerate(axes[1:]):
    ax.imshow(rotate(demo_img))
    ax.set_title(f'Rotated #{i+1}', fontsize=11, color=NAVY)
    ax.axis('off')
plt.tight_layout()
plt.show()

**Normalize: shift pixel values into the format the model expects**


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

resized   = transforms.CenterCrop(224)(transforms.Resize(256)(demo_img))
tensor    = to_tensor(resized)             # shape [3, 224, 224], values 0–1
normed    = normalize(tensor)              # values centered around 0

# Helper: undo normalization for display
def denormalize(t, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')

axes[0].imshow(resized)
axes[0].set_title('Resized image\n(0–255 pixels)', color=NAVY)
axes[0].axis('off')

axes[1].imshow(normed.permute(1, 2, 0).numpy())
axes[1].set_title(f'After Normalize\n(values: {normed.min():.2f} to {normed.max():.2f})',
                  color=NAVY)
axes[1].axis('off')

axes[2].imshow(denormalize(normed).permute(1, 2, 0).numpy())
axes[2].set_title('De-normalized\n(back to looking normal)', color=NAVY)
axes[2].axis('off')

plt.tight_layout()
plt.show()
print(f'Tensor shape: {tuple(normed.shape)}')
print(f'Normalized pixel range: [{normed.min():.3f}, {normed.max():.3f}]')

The full **training pipeline**:

```
RandomResizedCrop  →  HorizontalFlip  →  ColorJitter  →  Rotation  →  ToTensor  →  Normalize
```
The **evaluation pipeline**:

```
Resize(256)  →  CenterCrop(224)  →  ToTensor  →  Normalize
```

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Training pipeline:')
for t in train_transforms.transforms:
    print(f'  • {t.__class__.__name__}')

print('\nEvaluation pipeline:')
for t in eval_transforms.transforms:
    print(f'  • {t.__class__.__name__}')

**Full pipeline**:

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 11))
fig.patch.set_facecolor('white')
torch.manual_seed(42)
for i, ax in enumerate(axes.flat):
    out = train_transforms(demo_img)
    ax.imshow(denormalize(out).permute(1, 2, 0).numpy())
    ax.set_title(f'Augmented view #{i+1}', fontsize=10, color=NAVY)
    ax.axis('off')
plt.suptitle('Same dog, 12 augmented views — what the model sees during training',
             fontsize=15, fontweight='bold', color=NAVY, y=1.0)
plt.tight_layout()
plt.show()

**Wrap It In a PyTorch Dataset**

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class DogDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.classes = sorted(self.df['breed'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.class_to_idx[row['breed']]
        return img, label

train_ds = DogDataset(train_df, transform=train_transforms)
val_ds   = DogDataset(val_df,   transform=eval_transforms)
test_ds  = DogDataset(test_df,  transform=eval_transforms)

# Sanity check
img, lbl = train_ds[0]
print(f'Number of breeds:  {len(train_ds.classes)}')
print(f'Train size:        {len(train_ds):,}')
print(f'Sample tensor:     shape={tuple(img.shape)}, dtype={img.dtype}')
print(f'Sample label:      {lbl}  ({train_ds.classes[lbl]})')

**Handling Class Imbalance: Weighted Sampling**

In [ ]:
train_breed_counts = train_df['breed'].value_counts().to_dict()
sample_weights = train_df['breed'].map(lambda b: 1.0 / train_breed_counts[b]).values

# Show how this rebalances things
print(f'{"Breed":<25} {"Count":>6}   {"Weight":>10}   {"Relative":>10}')
print('-' * 60)
ref = max(train_breed_counts.values())
for b in ['Maltese Dog', 'Afghan Hound', 'Golden Retriever', 'Redbone', 'Pekinese']:
    if b in train_breed_counts:
        c = train_breed_counts[b]
        w = 1.0 / c
        rel = ref / c     # how many times more often this breed gets sampled vs the most common
        print(f'{b:<25} {c:>6}   {w:>10.5f}   {rel:>9.2f}×')

In [ ]:
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)
print(f'Sampler ready over {len(sample_weights):,} training samples.')

**DataLoaders: feed batches to the model**

In [ ]:
BATCH_SIZE  = 64
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)


imgs, lbls = next(iter(train_loader))
print(f'Batch shape:        {tuple(imgs.shape)}    (batch, channels, height, width)')
print(f'Pixel value range:  [{imgs.min():.2f}, {imgs.max():.2f}]   (negatives are correct after normalization)')
print(f'Unique breeds:      {len(set(lbls.tolist()))} / {BATCH_SIZE}   (high diversity = sampler working)')

**Double Checking**

In [ ]:
imgs, lbls = next(iter(train_loader))

fig, axes = plt.subplots(3, 4, figsize=(15, 11))
fig.patch.set_facecolor('white')
for ax, img, lbl in zip(axes.flat, imgs[:12], lbls[:12]):
    ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(train_ds.classes[lbl], fontsize=10, color=NAVY, fontweight='semibold')
    ax.axis('off')
plt.suptitle('Real training batch — 12 different dogs, fully augmented',
             fontsize=15, fontweight='bold', color=NAVY, y=1.0)
plt.tight_layout()
plt.show()

# **Question 1 — Multi-Task Learner for Dog Image Understanding**

The goal of this section: take a dog image and predict **four things at once** — breed, breed group, size, and coat length — using a single neural network with one shared backbone and four prediction heads

**Adding the three new labels**

In [ ]:
# ===== Breed → Group (AKC classification, 8 groups) =====
BREED_GROUPS = {
    'Sporting': [
        'Brittany Spaniel', 'Chesapeake Bay Retriever', 'Clumber',
        'Cocker Spaniel', 'Curly-Coated Retriever', 'English Setter',
        'English Springer', 'Flat-Coated Retriever',
        'German Short-Haired Pointer', 'Golden Retriever', 'Gordon Setter',
        'Irish Setter', 'Irish Water Spaniel', 'Labrador Retriever',
        'Sussex Spaniel', 'Vizsla', 'Weimaraner', 'Welsh Springer Spaniel',
    ],
    'Hound': [
        'Afghan Hound', 'Basenji', 'Basset', 'Beagle',
        'Black And Tan Coonhound', 'Bloodhound', 'Bluetick', 'Borzoi',
        'English Foxhound', 'Ibizan Hound', 'Irish Wolfhound',
        'Italian Greyhound', 'Norwegian Elkhound', 'Otterhound', 'Redbone',
        'Rhodesian Ridgeback', 'Saluki', 'Scottish Deerhound',
        'Walker Hound', 'Whippet',
    ],
    'Working': [
        'Appenzeller', 'Bernese Mountain Dog', 'Boxer', 'Bull Mastiff',
        'Doberman', 'Entlebucher', 'Eskimo Dog', 'Giant Schnauzer',
        'Great Dane', 'Great Pyrenees', 'Greater Swiss Mountain Dog',
        'Komondor', 'Kuvasz', 'Leonberg', 'Malamute', 'Newfoundland',
        'Rottweiler', 'Saint Bernard', 'Samoyed', 'Siberian Husky',
        'Standard Schnauzer', 'Tibetan Mastiff',
    ],
    'Terrier': [
        'Airedale', 'American Staffordshire Terrier', 'Australian Terrier',
        'Bedlington Terrier', 'Border Terrier', 'Cairn', 'Dandie Dinmont',
        'Irish Terrier', 'Kerry Blue Terrier', 'Lakeland Terrier',
        'Miniature Schnauzer', 'Norfolk Terrier', 'Norwich Terrier',
        'Scotch Terrier', 'Sealyham Terrier', 'Soft-Coated Wheaten Terrier',
        'Staffordshire Bullterrier', 'West Highland White Terrier',
        'Wire-Haired Fox Terrier',
    ],
    'Toy': [
        'Affenpinscher', 'Blenheim Spaniel', 'Brabancon Griffon',
        'Chihuahua', 'Japanese Spaniel', 'Maltese Dog', 'Miniature Pinscher',
        'Papillon', 'Pekinese', 'Pomeranian', 'Pug', 'Shih-Tzu',
        'Silky Terrier', 'Toy Poodle', 'Toy Terrier', 'Yorkshire Terrier',
    ],
    'Non-Sporting': [
        'Boston Bull', 'Chow', 'French Bulldog', 'Keeshond', 'Lhasa',
        'Mexican Hairless', 'Miniature Poodle', 'Schipperke',
        'Standard Poodle', 'Tibetan Terrier',
    ],
    'Herding': [
        'Border Collie', 'Bouvier Des Flandres', 'Briard', 'Cardigan',
        'Collie', 'German Shepherd', 'Groenendael', 'Kelpie', 'Malinois',
        'Old English Sheepdog', 'Pembroke', 'Shetland Sheepdog',
    ],
    'Miscellaneous': [
        'African Hunting Dog', 'Dhole', 'Dingo',
    ],
}

# ===== Breed → Size (5 categories, by typical adult weight) =====
BREED_SIZES = {
    'Toy': [
        'Affenpinscher', 'Blenheim Spaniel', 'Brabancon Griffon', 'Chihuahua',
        'Japanese Spaniel', 'Maltese Dog', 'Miniature Pinscher', 'Papillon',
        'Pekinese', 'Pomeranian', 'Shih-Tzu', 'Silky Terrier', 'Toy Poodle',
        'Toy Terrier', 'Yorkshire Terrier',
    ],
    'Small': [
        'Australian Terrier', 'Basenji', 'Beagle', 'Bedlington Terrier',
        'Border Terrier', 'Boston Bull', 'Cairn', 'Cocker Spaniel',
        'Dandie Dinmont', 'French Bulldog', 'Italian Greyhound',
        'Lakeland Terrier', 'Lhasa', 'Miniature Poodle', 'Miniature Schnauzer',
        'Norfolk Terrier', 'Norwich Terrier', 'Pug', 'Schipperke',
        'Scotch Terrier', 'Sealyham Terrier', 'Shetland Sheepdog',
        'Tibetan Terrier', 'West Highland White Terrier',
        'Wire-Haired Fox Terrier',
    ],
    'Medium': [
        'Airedale', 'American Staffordshire Terrier', 'Basset', 'Bluetick',
        'Border Collie', 'Brittany Spaniel', 'Cardigan',
        'Black And Tan Coonhound', 'Chow', 'Curly-Coated Retriever',
        'English Foxhound', 'English Springer', 'Ibizan Hound',
        'Irish Terrier', 'Keeshond', 'Kelpie', 'Kerry Blue Terrier',
        'Mexican Hairless', 'Norwegian Elkhound', 'Pembroke', 'Redbone',
        'Saluki', 'Soft-Coated Wheaten Terrier', 'Staffordshire Bullterrier',
        'Standard Poodle', 'Standard Schnauzer', 'Sussex Spaniel', 'Vizsla',
        'Walker Hound', 'Welsh Springer Spaniel', 'Whippet', 'Dingo',
        'Dhole', 'African Hunting Dog', 'Clumber', 'Appenzeller',
        'Entlebucher',
    ],
    'Large': [
        'Afghan Hound', 'Bloodhound', 'Borzoi', 'Boxer',
        'Bouvier Des Flandres', 'Briard', 'Chesapeake Bay Retriever',
        'Collie', 'Doberman', 'English Setter', 'Eskimo Dog',
        'Flat-Coated Retriever', 'German Shepherd',
        'German Short-Haired Pointer', 'Giant Schnauzer', 'Golden Retriever',
        'Gordon Setter', 'Groenendael', 'Irish Setter', 'Irish Water Spaniel',
        'Komondor', 'Kuvasz', 'Labrador Retriever', 'Malamute', 'Malinois',
        'Old English Sheepdog', 'Otterhound', 'Rhodesian Ridgeback',
        'Rottweiler', 'Samoyed', 'Siberian Husky', 'Weimaraner',
        'Bernese Mountain Dog',
    ],
    'Giant': [
        'Bull Mastiff', 'Great Dane', 'Great Pyrenees',
        'Greater Swiss Mountain Dog', 'Irish Wolfhound', 'Leonberg',
        'Newfoundland', 'Saint Bernard', 'Scottish Deerhound',
        'Tibetan Mastiff',
    ],
}

# ===== Breed → Coat Length (3 categories) =====
BREED_COATS = {
    'Short': [
        'American Staffordshire Terrier', 'Basenji', 'Beagle',
        'Black And Tan Coonhound', 'Bluetick', 'Boston Bull', 'Boxer',
        'Bull Mastiff', 'Chihuahua', 'Doberman', 'Dingo',
        'Dhole', 'English Foxhound', 'French Bulldog',
        'German Short-Haired Pointer', 'Great Dane', 'Greater Swiss Mountain Dog',
        'Italian Greyhound', 'Labrador Retriever', 'Mexican Hairless',
        'Miniature Pinscher', 'Pug', 'Redbone', 'Rhodesian Ridgeback',
        'Rottweiler', 'Saluki', 'Staffordshire Bullterrier',
        'Toy Terrier', 'Vizsla', 'Walker Hound', 'Weimaraner', 'Whippet',
        'African Hunting Dog', 'Appenzeller', 'Entlebucher', 'Ibizan Hound',
    ],
    'Medium': [
        'Airedale', 'Australian Terrier', 'Bedlington Terrier', 'Basset',
        'Border Collie', 'Border Terrier', 'Bloodhound', 'Brittany Spaniel',
        'Cairn', 'Cardigan', 'Cocker Spaniel', 'Curly-Coated Retriever',
        'Dandie Dinmont', 'English Setter', 'English Springer', 'Eskimo Dog',
        'Flat-Coated Retriever', 'German Shepherd', 'Giant Schnauzer',
        'Golden Retriever', 'Gordon Setter', 'Irish Setter', 'Irish Terrier',
        'Irish Water Spaniel', 'Japanese Spaniel', 'Kelpie',
        'Kerry Blue Terrier', 'Lakeland Terrier', 'Leonberg', 'Malamute',
        'Malinois', 'Miniature Schnauzer', 'Newfoundland', 'Norfolk Terrier',
        'Norwich Terrier', 'Norwegian Elkhound', 'Pembroke', 'Saint Bernard',
        'Samoyed', 'Schipperke', 'Scotch Terrier', 'Sealyham Terrier',
        'Siberian Husky', 'Soft-Coated Wheaten Terrier',
        'Standard Schnauzer', 'Sussex Spaniel', 'Tibetan Mastiff',
        'Toy Poodle', 'Welsh Springer Spaniel', 'West Highland White Terrier',
        'Wire-Haired Fox Terrier', 'Bernese Mountain Dog', 'Chesapeake Bay Retriever',
        'Bouvier Des Flandres', 'Clumber', 'Komondor',
    ],
    'Long': [
        'Affenpinscher', 'Afghan Hound', 'Blenheim Spaniel', 'Borzoi',
        'Brabancon Griffon', 'Briard', 'Chow', 'Collie', 'Great Pyrenees',
        'Groenendael', 'Irish Wolfhound', 'Keeshond', 'Kuvasz', 'Lhasa',
        'Maltese Dog', 'Miniature Poodle', 'Old English Sheepdog',
        'Otterhound', 'Papillon', 'Pekinese', 'Pomeranian', 'Scottish Deerhound',
        'Shetland Sheepdog', 'Shih-Tzu', 'Silky Terrier', 'Standard Poodle',
        'Tibetan Terrier', 'Yorkshire Terrier',
    ],
}

# ===== Build reverse lookups =====
breed_to_group = {b: g for g, breeds in BREED_GROUPS.items() for b in breeds}
breed_to_size  = {b: s for s, breeds in BREED_SIZES.items()  for b in breeds}
breed_to_coat  = {b: c for c, breeds in BREED_COATS.items()  for b in breeds}

# Coverage check
all_breeds = set(df['breed'].unique())
print('Breed coverage:')
print(f'  Group:  {sum(b in breed_to_group for b in all_breeds)} / {len(all_breeds)}')
print(f'  Size:   {sum(b in breed_to_size  for b in all_breeds)} / {len(all_breeds)}')
print(f'  Coat:   {sum(b in breed_to_coat  for b in all_breeds)} / {len(all_breeds)}')

missing = [b for b in all_breeds
           if b not in breed_to_group or b not in breed_to_size or b not in breed_to_coat]
if missing:
    print(f'\n⚠️  Missing from at least one mapping: {missing}')
else:
    print('\n✓ Every breed maps to all three attributes.')

**apply the labels to all three splits**

In [ ]:
for d in (train_df, val_df, test_df):
    d['breed_group'] = d['breed'].map(breed_to_group)
    d['size']        = d['breed'].map(breed_to_size)
    d['coat']        = d['breed'].map(breed_to_coat)

print('Sanity check on the train split:')
display(train_df[['breed', 'breed_group', 'size', 'coat']].head(10))

print('\nLabel distributions (train):')
for col in ['breed_group', 'size', 'coat']:
    print(f'\n--- {col} ---')
    print(train_df[col].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in zip(axes, ['breed_group', 'size', 'coat'],
                           ['Breed Group', 'Size', 'Coat Length']):
    counts = train_df[col].value_counts()
    ax.bar(range(len(counts)), counts.values, color=SKY_BLUE, edgecolor=NAVY)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=30, ha='right')
    ax.set_title(title)
    ax.set_ylabel('Number of Training Images')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 50, f'{v:,}', ha='center', fontsize=9, color=NAVY)

plt.tight_layout()
plt.show()

**Multi-Task Dataset**

A new `MultiTaskDogDataset` returns four labels per image instead of one. We'll reuse the `train_transforms` and `eval_transforms` we built earlier.

In [ ]:
# Fix the breed name typo
for d in (BREED_GROUPS, BREED_SIZES, BREED_COATS):
    for k, v in d.items():
        if 'Black And Tan Coonhound' in v:
            v[v.index('Black And Tan Coonhound')] = 'Black-And-Tan Coonhound'

# Rebuild reverse lookups
breed_to_group = {b: g for g, breeds in BREED_GROUPS.items() for b in breeds}
breed_to_size  = {b: s for s, breeds in BREED_SIZES.items()  for b in breeds}
breed_to_coat  = {b: c for c, breeds in BREED_COATS.items()  for b in breeds}

# Re-apply
for d in (train_df, val_df, test_df):
    d['breed_group'] = d['breed'].map(breed_to_group)
    d['size']        = d['breed'].map(breed_to_size)
    d['coat']        = d['breed'].map(breed_to_coat)

# Sanity check — should be 0 missing now
for col in ['breed_group', 'size', 'coat']:
    print(f'{col}: {train_df[col].isna().sum()} missing')

In [ ]:
class MultiTaskDogDataset(Dataset):
    """Returns (image, label_dict) where label_dict has keys:
    'breed', 'group', 'size', 'coat'."""

    def __init__(self, dataframe, transform=None,
                 breed_classes=None, group_classes=None,
                 size_classes=None, coat_classes=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

        # Allow passing in fixed class orders (so train/val/test agree)
        self.breed_classes = breed_classes or sorted(dataframe['breed'].unique())
        self.group_classes = group_classes or sorted(dataframe['breed_group'].unique())
        self.size_classes  = size_classes  or ['Toy', 'Small', 'Medium', 'Large', 'Giant']
        self.coat_classes  = coat_classes  or ['Short', 'Medium', 'Long']

        self.breed_to_idx = {c: i for i, c in enumerate(self.breed_classes)}
        self.group_to_idx = {c: i for i, c in enumerate(self.group_classes)}
        self.size_to_idx  = {c: i for i, c in enumerate(self.size_classes)}
        self.coat_to_idx  = {c: i for i, c in enumerate(self.coat_classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = {
            'breed': self.breed_to_idx[row['breed']],
            'group': self.group_to_idx[row['breed_group']],
            'size':  self.size_to_idx[row['size']],
            'coat':  self.coat_to_idx[row['coat']],
        }
        return img, labels

# Build the three datasets (using the train split's class order so all splits agree)
breed_classes = sorted(train_df['breed'].unique())
group_classes = sorted(train_df['breed_group'].unique())
size_classes  = ['Toy', 'Small', 'Medium', 'Large', 'Giant']
coat_classes  = ['Short', 'Medium', 'Long']

train_mt_ds = MultiTaskDogDataset(train_df, train_transforms,
                                  breed_classes, group_classes,
                                  size_classes, coat_classes)
val_mt_ds   = MultiTaskDogDataset(val_df,   eval_transforms,
                                  breed_classes, group_classes,
                                  size_classes, coat_classes)
test_mt_ds  = MultiTaskDogDataset(test_df,  eval_transforms,
                                  breed_classes, group_classes,
                                  size_classes, coat_classes)

# Quick check
img, lbls = train_mt_ds[0]
print(f'Image shape: {tuple(img.shape)}')
print(f'Labels: {lbls}')
print(f'Decoded: breed={breed_classes[lbls["breed"]]}, group={group_classes[lbls["group"]]}, '
      f'size={size_classes[lbls["size"]]}, coat={coat_classes[lbls["coat"]]}')

**DataLoaders with weighted sampling**

We need a custom `collate_fn` because each label is a dict, not a plain tensor.

In [ ]:
def multitask_collate(batch):
    images = torch.stack([b[0] for b in batch])
    labels = {
        'breed': torch.tensor([b[1]['breed'] for b in batch], dtype=torch.long),
        'group': torch.tensor([b[1]['group'] for b in batch], dtype=torch.long),
        'size':  torch.tensor([b[1]['size']  for b in batch], dtype=torch.long),
        'coat':  torch.tensor([b[1]['coat']  for b in batch], dtype=torch.long),
    }
    return images, labels

# Reuse the breed-weighted sampler we built earlier
train_mt_loader = DataLoader(train_mt_ds, batch_size=BATCH_SIZE,
                             sampler=train_sampler,
                             num_workers=NUM_WORKERS, pin_memory=True,
                             collate_fn=multitask_collate)
val_mt_loader   = DataLoader(val_mt_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True,
                             collate_fn=multitask_collate)
test_mt_loader  = DataLoader(test_mt_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True,
                             collate_fn=multitask_collate)

# Smoke test
imgs, lbls = next(iter(train_mt_loader))
print(f'Batch: images={tuple(imgs.shape)}')
for k, v in lbls.items():
    print(f'  {k}: {tuple(v.shape)} (range: {v.min().item()}–{v.max().item()})')

**The Multi-Task Model**

**Architecture:** one shared **ResNet-50 backbone** (pretrained on ImageNet) followed by **four small classification heads** — one per task.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class MultiTaskDogModel(nn.Module):
    def __init__(self, n_breed, n_group, n_size, n_coat,
                 backbone='resnet50', dropout=0.3):
        super().__init__()
        # Pretrained backbone
        if backbone == 'resnet50':
            base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
            feat_dim = base.fc.in_features
            base.fc = nn.Identity()
        elif backbone == 'resnet18':
            base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            feat_dim = base.fc.in_features
            base.fc = nn.Identity()
        else:
            raise ValueError(f'Unknown backbone: {backbone}')

        self.backbone = base
        self.dropout  = nn.Dropout(dropout)

        # One classification head per task
        self.breed_head = nn.Linear(feat_dim, n_breed)
        self.group_head = nn.Linear(feat_dim, n_group)
        self.size_head  = nn.Linear(feat_dim, n_size)
        self.coat_head  = nn.Linear(feat_dim, n_coat)

    def forward(self, x):
        feats = self.dropout(self.backbone(x))
        return {
            'breed': self.breed_head(feats),
            'group': self.group_head(feats),
            'size':  self.size_head(feats),
            'coat':  self.coat_head(feats),
        }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = MultiTaskDogModel(
    n_breed=len(breed_classes),
    n_group=len(group_classes),
    n_size=len(size_classes),
    n_coat=len(coat_classes),
    backbone='resnet50',
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

**Training Setup**

**Combined loss = breed_loss + 0.5 × group_loss + 0.5 × size_loss + 0.5 × coat_loss**

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Loss functions (CrossEntropy = standard for classification)
criterion = {k: nn.CrossEntropyLoss() for k in ['breed', 'group', 'size', 'coat']}

# Loss weights — breed is primary, others are auxiliary regularizers
LOSS_WEIGHTS = {'breed': 1.0, 'group': 0.5, 'size': 0.5, 'coat': 0.5}

# Different LRs for backbone (already trained) vs heads (random init)
backbone_params = list(model.backbone.parameters())
head_params = (list(model.breed_head.parameters()) +
               list(model.group_head.parameters()) +
               list(model.size_head.parameters())  +
               list(model.coat_head.parameters()))

optimizer = AdamW([
    {'params': backbone_params, 'lr': 1e-4},
    {'params': head_params,     'lr': 1e-3},
], weight_decay=1e-4)

EPOCHS = 8
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Loss weights: {LOSS_WEIGHTS}')
print(f'Backbone LR:  1e-4')
print(f'Head LR:      1e-3')
print(f'Epochs:       {EPOCHS}')

**The training loop**

In [ ]:
import time
from collections import defaultdict

def run_epoch(model, loader, optimizer=None, desc=''):
    """One pass over `loader`. If optimizer is None, runs in eval mode."""
    train_mode = optimizer is not None
    model.train(train_mode)
    totals = defaultdict(float)   # sum of losses
    correct = defaultdict(int)
    n = 0

    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = {k: v.to(device, non_blocking=True) for k, v in lbls.items()}

        with torch.set_grad_enabled(train_mode):
            outs = model(imgs)
            losses = {k: criterion[k](outs[k], lbls[k]) for k in outs}
            loss = sum(LOSS_WEIGHTS[k] * losses[k] for k in losses)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        bs = imgs.size(0)
        n += bs
        totals['total'] += loss.item() * bs
        for k in outs:
            totals[k] += losses[k].item() * bs
            correct[k] += (outs[k].argmax(1) == lbls[k]).sum().item()

    metrics = {f'{k}_loss': totals[k] / n for k in totals}
    for k in ('breed', 'group', 'size', 'coat'):
        metrics[f'{k}_acc'] = correct[k] / n
    return metrics

# ===== Run training =====
history = []
best_val_breed_acc = 0.0
best_state = None

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    et = time.time()
    train_m = run_epoch(model, train_mt_loader, optimizer, desc=f'train e{epoch}')
    val_m   = run_epoch(model, val_mt_loader,   None,      desc=f'val   e{epoch}')
    scheduler.step()

    history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_m.items()},
                                    **{f'val_{k}':   v for k, v in val_m.items()}})

    print(f'Epoch {epoch:2d}/{EPOCHS}  '
          f'[{time.time()-et:5.1f}s]  '
          f'train: loss={train_m["total_loss"]:.3f} breed={train_m["breed_acc"]:.3f}  '
          f'val: loss={val_m["total_loss"]:.3f} breed={val_m["breed_acc"]:.3f} '
          f'group={val_m["group_acc"]:.3f} size={val_m["size_acc"]:.3f} '
          f'coat={val_m["coat_acc"]:.3f}')

    # Save best (by val breed accuracy)
    if val_m['breed_acc'] > best_val_breed_acc:
        best_val_breed_acc = val_m['breed_acc']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'         ✓ new best (val breed acc = {best_val_breed_acc:.3f}) — saving')

print(f'\nTotal training time: {(time.time() - t0)/60:.1f} min')
print(f'Best val breed acc: {best_val_breed_acc:.3f}')

# Load best weights for evaluation / demo
model.load_state_dict(best_state)
torch.save(best_state, 'multitask_resnet50_best.pt')
print('Saved best weights → multitask_resnet50_best.pt')

**Training curves**

In [ ]:
hist_df = pd.DataFrame(history).set_index('epoch')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(hist_df.index, hist_df['train_total_loss'], color=SKY_BLUE,
             marker='o', linewidth=2, label='Train')
axes[0].plot(hist_df.index, hist_df['val_total_loss'], color=NAVY,
             marker='s', linewidth=2, label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Combined Loss')
axes[0].set_title('Loss over Epochs'); axes[0].legend()

# Per-task validation accuracy
for k, c in zip(['breed', 'group', 'size', 'coat'],
                [SKY_BLUE, STEEL_BLUE, ACCENT, '#1B4F72']):
    axes[1].plot(hist_df.index, hist_df[f'val_{k}_acc'], color=c,
                 marker='o', linewidth=2, label=k.title())
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Per-Task Validation Accuracy'); axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

**Test Set Evaluation**

In [ ]:
test_m = run_epoch(model, test_mt_loader, None)
print('Held-out test set performance:')
print('-' * 40)
print(f'  Breed:        {test_m["breed_acc"]:.3f}')
print(f'  Breed Group:  {test_m["group_acc"]:.3f}')
print(f'  Size:         {test_m["size_acc"]:.3f}')
print(f'  Coat Length:  {test_m["coat_acc"]:.3f}')

**Per-class breed accuracy (top and bottom 10)**

In [ ]:
# Collect all predictions on the test set
model.eval()
all_breed_pred, all_breed_true = [], []
with torch.no_grad():
    for imgs, lbls in test_mt_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        all_breed_pred.extend(out['breed'].argmax(1).cpu().tolist())
        all_breed_true.extend(lbls['breed'].tolist())

per_class = pd.DataFrame({'true': all_breed_true, 'pred': all_breed_pred})
per_class['breed'] = per_class['true'].map(lambda i: breed_classes[i])
per_class['correct'] = per_class['true'] == per_class['pred']

acc_per_breed = per_class.groupby('breed')['correct'].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
fig.patch.set_facecolor('white')

# Bottom 10
bot = acc_per_breed.head(10)
axes[0].barh(range(len(bot)), bot.values, color=LIGHT_BLUE, edgecolor=NAVY)
axes[0].set_yticks(range(len(bot))); axes[0].set_yticklabels(bot.index)
axes[0].invert_yaxis(); axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Hardest 10 Breeds'); axes[0].set_xlim(0, 1)
for i, v in enumerate(bot.values):
    axes[0].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9, color=NAVY)

# Top 10
top = acc_per_breed.tail(10).iloc[::-1]
axes[1].barh(range(len(top)), top.values, color=SKY_BLUE, edgecolor=NAVY)
axes[1].set_yticks(range(len(top))); axes[1].set_yticklabels(top.index)
axes[1].invert_yaxis(); axes[1].set_xlabel('Test Accuracy')
axes[1].set_title('Easiest 10 Breeds'); axes[1].set_xlim(0, 1)
for i, v in enumerate(top.values):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9, color=NAVY)

plt.tight_layout()

**Gradio Demo**

In [ ]:
!pip install gradio --quiet

In [ ]:
import gradio as gr
import numpy as np

# Move the model to eval mode for inference
model.eval()

def predict_dog(image):
    if image is None:
        return {}, '—', '—', '—'

    # Gradio gives us a numpy array — convert to PIL for our transforms
    img = Image.fromarray(image).convert('RGB')
    tensor = eval_transforms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(tensor)

    # Top-5 breed predictions
    breed_probs = F.softmax(out['breed'], dim=1)[0]
    top5 = torch.topk(breed_probs, k=5)
    top5_breeds = {
        breed_classes[i.item()]: float(p.item())
        for p, i in zip(top5.values, top5.indices)
    }

    # Single-best for the auxiliary tasks
    pred_group = group_classes[out['group'].argmax(1).item()]
    pred_size  = size_classes[out['size'].argmax(1).item()]
    pred_coat  = coat_classes[out['coat'].argmax(1).item()]

    return top5_breeds, pred_group, pred_size, pred_coat

# Build the interface
interface = gr.Interface(
    fn=predict_dog,
    inputs=gr.Image(type='numpy', label='Upload a dog photo'),
    outputs=[
        gr.Label(num_top_classes=5, label='Top 5 Breed Predictions'),
        gr.Textbox(label='Breed Group'),
        gr.Textbox(label='Size'),
        gr.Textbox(label='Coat Length'),
    ],
    title='🐕 PupMatch — Dog Image Analyzer',
    description=('Upload any dog photo and the multi-task model will predict its breed, '
                 'breed group, size, and coat length. Trained on Stanford Dogs (120 breeds).'),
    flagging_mode='never',
)

# share=True gives you a public URL that works anywhere
interface.launch(share=True, debug=False)